In [1]:
import os
import csv
from tqdm import tqdm

def stream_merge_cluster_cossims(cluster_labels, dataset_name, cossim_file, scratch_dir, output_file):
    """
    Streams rows from individual per-cluster cosine similarity CSVs into a single file with
    columns = cluster labels, and rows = samples.
    """
    os.makedirs(os.path.dirname(output_file), exist_ok=True)

    input_files = []
    readers = []
    headers = []

    # Open all files, collect readers and headers
    for cluster_label in cluster_labels:
        filepath = f'{scratch_dir}/Cosine_Similarities/{dataset_name}/kmeans/{cluster_label}_{cossim_file}'
        if not os.path.exists(filepath):
            print(f"[Warning] Missing file: {filepath}")
            continue
        f = open(filepath, 'r')
        reader = csv.reader(f)
        header = next(reader)  # Skip header
        input_files.append(f)
        readers.append(reader)
        headers.append(str(cluster_label))

    # Estimate total rows from first file
    try:
        row_count = sum(1 for _ in open(f'{scratch_dir}/Cosine_Similarities/{dataset_name}/kmeans/{cluster_labels[0]}_{cossim_file}')) - 1
    except Exception:
        row_count = None

    with open(output_file, 'w', newline='') as out_f:
        writer = csv.writer(out_f)
        writer.writerow(headers)  # Write cluster labels as column headers

        for _ in tqdm(range(row_count) if row_count else iter(int, 1), desc="Merging rows", unit="rows"):
            try:
                row = [next(reader)[0] for reader in readers]
                writer.writerow(row)
            except StopIteration:
                break

    for f in input_files:
        f.close()

    print(f"[✔] Merged {len(readers)} cluster files into: {output_file}")


In [2]:
cluster_labels = range(1000)
dataset_name = 'Coco'
MODEL_NAME = 'Llama'
CONCEPTS_FILE = f'kmeans_1000_concepts_{MODEL_NAME}_patch_embeddings_percentthrumodel_100.pt'
COSSIM_FILE = f'cosine_similarities_{CONCEPTS_FILE[:-3]}.csv'
scratch_dir = '/scratch/cgoldberg'
output_file = f'{scratch_dir}/Cosine_Similarities/{dataset_name}/kmeans/allpairs_{COSSIM_FILE}'

stream_merge_cluster_cossims(cluster_labels, dataset_name, COSSIM_FILE, scratch_dir, output_file)

Merging rows:   0%|          | 29437/8000000 [00:15<1:08:46, 1931.40rows/s]


KeyboardInterrupt: 

In [4]:
import pandas as pd
dff = pd.read_csv(f'{scratch_dir}/Cosine_Similarities/{dataset_name}/kmeans/allpairs_{COSSIM_FILE}')
print(dff)

              0         1         2         3         4         5         6  \
0      0.027935 -0.059796  0.028902 -0.002315  0.104260 -0.047860 -0.070558   
1      0.073169 -0.056073  0.033447  0.008895 -0.004380 -0.045085 -0.066141   
2      0.097165 -0.085466  0.021922 -0.032642 -0.046838 -0.007810 -0.066629   
3      0.016822 -0.052101 -0.030896 -0.051592  0.018572 -0.017982 -0.015265   
4     -0.035928 -0.041026 -0.021332 -0.108505 -0.031232 -0.012979 -0.074195   
...         ...       ...       ...       ...       ...       ...       ...   
29432  0.324185 -0.059396  0.006800  0.129046 -0.038130  0.142051 -0.074755   
29433 -0.043159  0.013715 -0.045686 -0.018408  0.263185  0.032300  0.005716   
29434 -0.033147  0.005750  0.120796 -0.007187  0.021469 -0.029785 -0.000499   
29435 -0.038525 -0.004099  0.093124  0.009905 -0.034753 -0.039113  0.001801   
29436  0.076956 -0.006422  0.068034  0.067325 -0.011659  0.082681 -0.023965   

              7         8         9  ...       990 

In [7]:
import os
import torch
import pandas as pd
import pyarrow

def convert_cossim_df_to_tensor_inplace(input_csv_path, output_tensor_name=None, chunk_size=100, device='cpu'):
    """
    Converts a large cosine similarity CSV [n_samples, n_clusters] to a tensor [n_clusters, n_samples]
    by incrementally writing to a preallocated tensor (no chunk tracking needed).

    Args:
        input_csv_path (str): Path to cosine similarity CSV.
        output_tensor_name (str, optional): Output .pt filename. If None, inferred from input.
        chunk_size (int): Number of columns to load at a time.
        device (str): 'cuda' or 'cpu' to allocate tensor.
    """
    print(f"[Start] Converting {input_csv_path} to tensor (in-place)")

    # Get column names and number of rows
    all_columns = pd.read_csv(input_csv_path, nrows=0).columns.tolist()
    n_clusters = len(all_columns)

    # Use just one column to count number of rows
    n_samples = len(pd.read_csv(input_csv_path, usecols=[all_columns[0]]))

    # Preallocate final tensor [n_clusters, n_samples]
    final_tensor = torch.empty((n_clusters, n_samples), dtype=torch.float32, device=device)

    # Process in chunks (across columns)
    for start in tqdm(range(0, n_clusters, chunk_size), desc="Writing chunks"):
        end = min(start + chunk_size, n_clusters)
        chunk_cols = all_columns[start:end]

        df_chunk = pd.read_csv(input_csv_path, usecols=chunk_cols)
        chunk_tensor = torch.tensor(df_chunk.values.T, dtype=torch.float32, device=device)  # [chunk_size, n_samples]
        final_tensor[start:end] = chunk_tensor  # insert in-place

        del df_chunk, chunk_tensor
        torch.cuda.empty_cache()

    # Save tensor
    output_dir = os.path.dirname(input_csv_path)
    if output_tensor_name is None:
        base = os.path.splitext(os.path.basename(input_csv_path))[0]
        output_tensor_name = f"{base}.pt"
    output_path = os.path.join(output_dir, output_tensor_name)

    torch.save(final_tensor, output_path)
    print(f"[Saved] Tensor shape: {final_tensor.shape} -> {output_path}")

In [ ]:
dataset_name = 'Coco'
MODEL_NAME = 'Llama'
CONCEPTS_FILE = f'kmeans_1000_concepts_{MODEL_NAME}_patch_embeddings_percentthrumodel_100.pt'
COSSIM_FILE = f'cosine_similarities_{CONCEPTS_FILE[:-3]}.csv'
scratch_dir = '/scratch/cgoldberg'
output_file = f'{scratch_dir}/Cosine_Similarities/{dataset_name}/kmeans/{COSSIM_FILE}'
convert_cossim_df_to_tensor(output_file, output_tensor_name=None)

[Loading] Reading CSV from: /scratch/cgoldberg/Cosine_Similarities/Coco/kmeans/cosine_similarities_kmeans_1000_concepts_Llama_patch_embeddings_percentthrumodel_100.csv


In [ ]:
dataset_name = 'Coco'
MODEL_NAME = 'Llama'
CONCEPTS_FILE = f'kmeans_1000_concepts_{MODEL_NAME}_patch_embeddings_percentthrumodel_100.pt'
DISTS_FILE = f'dists_{CONCEPTS_FILE[:-3]}.csv'
scratch_dir = '/scratch/cgoldberg'
dists_file = f'{scratch_dir}/DISTANCES/{dataset_name}/kmeans/{DISTS_FILE}'
convert_cossim_df_to_tensor(dists_file, output_tensor_name=None)